# 6장 — Transformer 블록 → GPT 조립 (실습)

교재 `docs/book/06-transformer-gpt.md` 와 함께 본다. 이 노트북에서 하는 것:

1. LayerNorm 과 잔차 연결이 각각 무엇을 하는지 숫자로 본다
2. 블록을 쌓아 GPT 를 만들고 `(B, T) → (B, T, V)` 흐름과 파라미터 수를 센다
3. 짧게 학습한 뒤 어텐션 맵과 생성문을 본다 (본 학습은 7장)

> 전체 실행 약 3분.

## 1. LayerNorm — 벡터마다 눈금 맞추기

In [ ]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from shllm.config import TOKENIZER_DIR, setup_cpu

setup_cpu()
x = torch.tensor([[1.0, 2.0, 3.0, 10.0], [0.1, 0.2, 0.3, 0.4]])  # 토큰 벡터 2개 (C=4)
ln = nn.LayerNorm(4)
y = ln(x)
print("입력 행 평균/표준편차:", x.mean(1).tolist(), [round(v, 3) for v in x.std(1, unbiased=False).tolist()])
print("출력 행 평균/표준편차:", [round(v, 3) for v in y.mean(1).tolist()], [round(v, 3) for v in y.std(1, unbiased=False).tolist()])
print("학습되는 파라미터: gamma", ln.weight.data.tolist(), "beta", ln.bias.data.tolist())

LayerNorm 은 **토큰 벡터 하나(C 차원) 안에서** 평균 0·표준편차 1 로 맞춘 뒤 학습되는 배율(γ)·이동(β)을 곱하고 더한다.
배치나 다른 토큰과 무관하게 각 벡터 스스로 정규화하므로 문장 길이·배치 크기에 영향받지 않는다. 층을 거칠수록 값의 크기가 제멋대로 커지는 것을 막아 깊은 모델을 학습 가능하게 한다.

## 2. 잔차 연결 — 기울기의 고속도로

In [ ]:
def deep_net(n_layers: int, residual: bool):
    torch.manual_seed(0)
    layers = [nn.Sequential(nn.Linear(32, 32), nn.Tanh()) for _ in range(n_layers)]
    x = torch.randn(8, 32, requires_grad=True)
    h = x
    for layer in layers:
        h = h + layer(h) if residual else layer(h)
    h.sum().backward()
    return x.grad.abs().mean().item()  # 입력까지 기울기가 얼마나 살아 도착했나


for L in (2, 8, 32):
    print(f"{L:>2}층: 입력 기울기 크기  잔차 없음 {deep_net(L, False):.2e}   잔차 있음 {deep_net(L, True):.2e}")

잔차 없이 tanh 층을 32개 쌓으면 입력에 도착하는 기울기가 사실상 0 이다(기울기 소실). `x + f(x)` 로 쓰면 미분에 항상 항등항 1 이 있어 기울기가 층을 건너뛰어 흐른다.
GPT 의 블록은 `x = x + attn(ln(x)); x = x + mlp(ln(x))` — 각 부품은 x 를 "얼마나 고칠지" 만 배운다.

## 3. GPT 조립

In [ ]:
from shllm.model import GPT, GPTConfig
from shllm.tokenizer import BPETokenizer

tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=64, n_layer=2, n_head=4, n_embd=128)
model = GPT(cfg)
print(model)

In [ ]:
idx = torch.tensor([tok.encode("옛날 옛적에 호랑이가 담배 피우던 시절에")])
logits, loss = model(idx, idx)  # targets 로 자기 자신을 넣으면 "각 자리에서 그 자리 토큰을 맞히나" — 모양 확인용
print("idx", tuple(idx.shape), "→ logits", tuple(logits.shape), "= (B, T, V)")
print("초기 loss", round(loss.item(), 3), "≈ log V =", round(torch.log(torch.tensor(float(tok.vocab_size))).item(), 3), "(아직 아무것도 모른다)")

### 3.1 파라미터는 어디에 있나

In [ ]:
def count(cfg: GPTConfig) -> dict:
    m = GPT(cfg)
    C, L = cfg.n_embd, cfg.n_layer
    return {
        "총계(위치 제외)": m.n_params(),
        "토큰 임베딩 V·C": cfg.vocab_size * C,
        "블록 합계 ≈ 12·C²·L": sum(p.numel() for p in m.blocks.parameters()),
        "  └ 어텐션 4C²": sum(p.numel() for b in m.blocks for p in b.attn.parameters()),
        "  └ MLP 8C²": sum(p.numel() for b in m.blocks for p in b.mlp.parameters()),
    }


for name, c in (
    ("노트북 tiny (2층·128)", GPTConfig(block_size=64, n_layer=2, n_head=4, n_embd=128)),
    ("small-cpu (6층·256)", GPTConfig(block_size=256, n_layer=6, n_head=8, n_embd=256)),
    ("GPT-2 small (12층·768)", GPTConfig(vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768)),
):
    print(f"[{name}]")
    for k, v in count(c).items():
        print(f"  {k:<22}{v:>14,}")

GPT-2 small 과 우리 `small-cpu` 는 **구조가 같고 숫자만 다르다**. 블록 파라미터는 `12·C²` 에 층 수를 곱한 것이고, 어휘 임베딩은 `V·C` 다.
우리 모델은 어휘가 작고(8,192) 폭이 좁아(256) 690만 개 — CPU 로 수 시간에 학습할 수 있는 크기다.

## 4. 짧게 학습해 보기 (600 스텝)

In [ ]:
from shllm.data import get_batch, load_corpus
from shllm.mlp import train_steps

text = load_corpus("korean-classics")
data = torch.tensor(tok.encode(text))
n = int(0.9 * len(data))
train, val = data[:n], data[n:]

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = GPT(cfg)
t0 = time.perf_counter()
losses = train_steps(model, lambda: get_batch(train, cfg.block_size, 32, g), steps=600, lr=1e-3, log_every=200)
with torch.no_grad():
    model.eval()
    vx, vy = get_batch(val, cfg.block_size, 64, torch.Generator().manual_seed(1))
    _, vl = model(vx, vy)
print(f"{time.perf_counter() - t0:.0f}초  train {sum(losses[-50:]) / 50:.3f}  val {vl.item():.3f}   (5장 어텐션 한 층·문맥 8: 약 7.0)")

### 4.1 어텐션 맵 — 학습 후 무엇을 보는가

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

sample = "그는 아버지의 집으로 돌아가서 어머니를 만났다."
ids = torch.tensor([tok.encode(sample)])
with torch.no_grad():
    model(ids)
maps = model.attention_maps()  # 층별 (B, 헤드, T, T)
labels = [tok.token_str(i).replace(" ", "␣") for i in ids[0].tolist()]
T = len(labels)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for layer in range(2):
    for head in range(4):
        ax = axes[layer, head]
        ax.imshow(maps[layer][0, head], cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"층 {layer} 헤드 {head}", fontsize=10)
        ax.set_xticks(range(T), labels, rotation=90, fontsize=7)
        ax.set_yticks(range(T), labels, fontsize=7)
plt.tight_layout()
plt.show()

행 = 보는 자리, 열 = 보이는 자리. 상삼각이 0 인 것은 인과 마스크. 헤드마다 패턴이 다르다 — 직전 토큰만 보는 헤드, 첫 토큰(문장 시작)에 몰리는 헤드, 넓게 퍼지는 헤드.
600 스텝짜리라 아직 거칠다. 7장 학습 후 같은 그림을 대시보드에서 본다.

### 4.2 생성 미리보기

In [ ]:
from shllm.generate import generate

g = torch.Generator().manual_seed(1337)
out = generate(model, torch.tensor([tok.encode("옛날 옛적에")]), max_new_tokens=60, temperature=1.0, generator=g)
print(tok.decode(out[0].tolist()))

## 정리

- 블록 = `x + attn(ln1(x))` 그리고 `x + mlp(ln2(x))`. LayerNorm 은 눈금 맞추기, 잔차는 기울기 고속도로, MLP 는 자리별 가공.
- GPT = 임베딩 → 블록 × L → LayerNorm → 어휘 점수. 입력 `(B, T)` 에서 **모든 자리의** 다음 토큰 점수 `(B, T, V)` 가 한 번에 나온다.
- 파라미터 ≈ `V·C + 12·C²·L`. GPT-2 와 같은 식, 숫자만 다르다.

---
**다음 장**: 7장 — 이 모델을 몇 시간 동안 제대로 학습시킨다.